In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# train_conditional_ddpm_x0.py
# 条件扩散：输入 noisy(x_t) + 漂移图 cond，预测 x0（原图）
# 每个 epoch 结束在训练/测试集上各保存一张对比图
# 依赖：torch, torchvision, pillow, numpy, tqdm

import os, math, glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, utils as vutils
from tqdm import tqdm
import matplotlib.pyplot as plt

# ===== Color & Edge utilities =====
def srgb_to_lab(img):  # img: [-1,1]
    eps = 1e-12
    x = (img.clamp(-1,1) + 1) * 0.5  # [0,1]
    thr = 0.04045
    linear = torch.where(x <= thr, x/12.92, ((x+0.055)/1.055).pow(2.4))
    r, g, b = linear[:,0], linear[:,1], linear[:,2]
    X = 0.4124564*r + 0.3575761*g + 0.1804375*b
    Y = 0.2126729*r + 0.7151522*g + 0.0721750*b
    Z = 0.0193339*r + 0.1191920*g + 0.9503041*b
    Xn, Yn, Zn = 0.95047, 1.0, 1.08883
    x = (X/(Xn+eps)).clamp(min=eps); y = (Y/(Yn+eps)).clamp(min=eps); z = (Z/(Zn+eps)).clamp(min=eps)
    d = 6/29
    def f(t): return torch.where(t > d**3, t.pow(1/3), t/(3*d**2) + 4/29)
    fx, fy, fz = f(x), f(y), f(z)
    L = 116*fy - 16; a = 500*(fx - fy); b = 200*(fy - fz)
    return torch.stack([L, a, b], dim=1)

def delta_e_ciede2000(lab1, lab2):  # returns [B,1,H,W]
    eps = 1e-12
    L1,a1,b1 = lab1[:,0], lab1[:,1], lab1[:,2]
    L2,a2,b2 = lab2[:,0], lab2[:,1], lab2[:,2]
    C1 = torch.sqrt(a1*a1 + b1*b1 + eps); C2 = torch.sqrt(a2*a2 + b2*b2 + eps)
    Cm = 0.5*(C1+C2)
    G = 0.5*(1 - torch.sqrt((Cm**7)/(Cm**7 + 25**7 + eps)))
    a1p=(1+G)*a1; a2p=(1+G)*a2
    C1p=torch.sqrt(a1p*a1p + b1*b1 + eps); C2p=torch.sqrt(a2p*a2p + b2*b2 + eps)
    def atan2d(y,x):
        ang = torch.atan2(y,x)*(180/math.pi);
        return torch.where(ang<0, ang+360, ang)
    h1p = torch.where((b1.abs()<eps)&(a1p.abs()<eps), torch.zeros_like(b1), atan2d(b1,a1p))
    h2p = torch.where((b2.abs()<eps)&(a2p.abs()<eps), torch.zeros_like(b2), atan2d(b2,a2p))
    dLp = L2-L1; dCp = C2p-C1p
    dhp = h2p-h1p; dhp = torch.where(dhp>180, dhp-360, torch.where(dhp<-180, dhp+360, dhp))
    dHp = 2*torch.sqrt(C1p*C2p + eps) * torch.sin((dhp*0.5)*(math.pi/180))
    Lpm=0.5*(L1+L2); Cpm=0.5*(C1p+C2p)
    hp_sum=h1p+h2p
    hpm = torch.where((C1p*C2p).le(eps), hp_sum,
           torch.where(hp_sum>360, (hp_sum-360)*0.5, torch.where(hp_sum<0,(hp_sum+360)*0.5,hp_sum*0.5)))
    T = 1 - 0.17*torch.cos((hpm-30)*(math.pi/180)) + 0.24*torch.cos((2*hpm)*(math.pi/180)) \
        + 0.32*torch.cos((3*hpm+6)*(math.pi/180)) - 0.20*torch.cos((4*hpm-63)*(math.pi/180))
    d_ro = 30*torch.exp(-((hpm-275)/25)**2)
    Rc = 2*torch.sqrt((Cpm**7)/(Cpm**7 + 25**7 + eps))
    Sl = 1 + (0.015*((Lpm-50)**2))/torch.sqrt(20+(Lpm-50)**2 + eps)
    Sc = 1 + 0.045*Cpm
    Sh = 1 + 0.015*Cpm*T
    Rt = -torch.sin(2*(d_ro)*(math.pi/180)) * Rc
    dE = torch.sqrt((dLp/Sl)**2 + (dCp/Sc)**2 + (dHp/Sh)**2 + Rt*(dCp/Sc)*(dHp/Sh) + eps)
    return dE.unsqueeze(1)

# Sobel 梯度（逐通道）
_SOBEL_X = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
_SOBEL_Y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
def grad_mag(x):  # x: [-1,1], [B,3,H,W]
    kx = _SOBEL_X.to(x.device).repeat(3,1,1,1)
    ky = _SOBEL_Y.to(x.device).repeat(3,1,1,1)
    gx = F.conv2d(x, kx, padding=1, groups=3)
    gy = F.conv2d(x, ky, padding=1, groups=3)
    return torch.sqrt(gx*gx + gy*gy + 1e-8)

# ===== 新增：简单 GN 残块（无时间）用于条件编码器 =====
def _gn_groups(c):
    g = min(32, c)
    while c % g != 0 and g > 1:
        g //= 2
    return g

class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.norm = nn.GroupNorm(_gn_groups(in_c), in_c)
        self.act  = nn.SiLU()
        self.conv = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.skip = nn.Identity() if in_c == out_c else nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        h = self.conv(self.act(self.norm(x)))
        return h if isinstance(self.skip, nn.Identity) else h  # 预激活一层即可

class CondEncoder(nn.Module):
    """
    把 cond (B,3,H,W) 编为与主干各层对齐的多尺度特征，通道数按 chans 对齐
    """
    def __init__(self, in_ch=3, chans=(64,128,256,512)):
        super().__init__()
        self.stem = nn.Conv2d(in_ch, chans[0], 3, padding=1)
        downs, blocks = [], []
        in_c = chans[0]
        for i, c in enumerate(chans):
            blocks.append(nn.Sequential(
                ConvGNAct(in_c, c),
                ConvGNAct(c, c),
            ))
            in_c = c
            if i < len(chans) - 1:
                downs.append(nn.Conv2d(in_c, in_c, 4, stride=2, padding=1))
        self.blocks = nn.ModuleList(blocks)
        self.downs  = nn.ModuleList(downs)

    def forward(self, x):
        feats = []
        x = self.stem(x)
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            feats.append(x)
            if i < len(self.blocks) - 1:
                x = self.downs[i](x)
        return feats  # len = L，与主干层级一致，空间分辨率匹配


# -----------------------------
# 实用函数
# -----------------------------
def to_uint8(img_tensor):
    # [-1,1] -> uint8
    img = (img_tensor.clamp(-1, 1) + 1) * 0.5
    img = (img * 255.0).round().clamp(0, 255).to(torch.uint8)
    return img

# -----------------------------
# 数据集：同目录内 <prefix>_orig.png / <prefix>_recon.png
# -----------------------------
class PairImageDataset(Dataset):
    def __init__(self, root="data", size=512,
                 orig_suffix="_orig.png", recon_suffix="_recon.png"):
        """
        root 目录下放置所有成对图片，例如：
            region_50_orig.png
            region_50_recon.png
        """
        self.size = size
        self.orig_suffix = orig_suffix
        self.recon_suffix = recon_suffix

        # 找到所有 *_orig.png，并匹配 *_recon.png
        orig_paths = sorted(glob.glob(os.path.join(root, f"*{orig_suffix}")))
        self.pairs = []
        for op in orig_paths:
            prefix = os.path.basename(op)[:-len(orig_suffix)]  # e.g. "region_50"
            rp = os.path.join(root, f"{prefix}{recon_suffix}")
            if os.path.exists(rp):
                # cond=recon（漂移图），gt=orig（原图）
                self.pairs.append((rp, op))
            else:
                print(f"[WARN] 找不到配对的重建图: {rp}")

        assert len(self.pairs) > 1, "请至少放置 2 对 *_orig.png / *_recon.png 文件以进行训练/测试划分"

        self.tf = transforms.Compose([
            transforms.ToTensor(),                      # [0,1]
            transforms.Normalize([0.5]*3, [0.5]*3)     # -> [-1,1]
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        recon_path, orig_path = self.pairs[idx]
        img_recon = Image.open(recon_path).convert("RGB")  # 漂移图（条件）
        img_orig  = Image.open(orig_path).convert("RGB")   # 原图（监督）

        d = self.tf(img_recon)  # cond
        o = self.tf(img_orig)   # x0 (gt)
        return d, o

def create_data_loaders(root, size, batch_size, num_workers, test_ratio=0.1, seed=42):
    """
    返回 train_loader, test_loader（9:1 默认）
    """
    fullset = PairImageDataset(root=root, size=size)
    n = len(fullset)

    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    n_test = max(1, int(round(n * test_ratio)))
    test_idx = idx[:n_test].tolist()
    train_idx = idx[n_test:].tolist()
    assert len(train_idx) > 0, "训练集为空，请增大数据或调小 test_ratio"

    train_set = Subset(fullset, train_idx)
    test_set  = Subset(fullset, test_idx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True, drop_last=False)
    return train_loader, test_loader

# -----------------------------
# 时间嵌入
# -----------------------------
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        # t: [B] -> [B, dim]
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=device) / (half - 1))
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0,1))
        return emb

class TimeMLP(nn.Module):
    def __init__(self, dim, out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, out),
            nn.SiLU(),
            nn.Linear(out, out)
        )
    def forward(self, t_emb):
        return self.net(t_emb)

# -----------------------------
# UNet 组件
# -----------------------------
def conv3x3(in_c, out_c):
    return nn.Conv2d(in_c, out_c, 3, padding=1)

class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, time_dim):
        super().__init__()
        g1 = min(32, in_c)
        g2 = min(32, out_c)
        # 保证分组数能整除通道数
        while in_c % g1 != 0: g1 //= 2
        while out_c % g2 != 0: g2 //= 2

        self.norm1 = nn.GroupNorm(g1, in_c)
        self.act1  = nn.SiLU()
        self.conv1 = conv3x3(in_c, out_c)
        self.norm2 = nn.GroupNorm(g2, out_c)
        self.act2  = nn.SiLU()
        self.conv2 = conv3x3(out_c, out_c)
        self.time  = nn.Linear(time_dim, out_c)
        self.skip  = nn.Identity() if in_c == out_c else nn.Conv2d(in_c, out_c, 1)

    def forward(self, x, t_emb):
        h = self.conv1(self.act1(self.norm1(x)))
        h = h + self.time(t_emb)[:, :, None, None]
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.op = nn.Conv2d(c, c, 4, stride=2, padding=1)
    def forward(self, x): return self.op(x)

class Up(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.op = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="nearest"),
            conv3x3(c, c)
        )
    def forward(self, x): return self.op(x)


# ===== 新增：二维 Cross-Attention（多头） =====
class CrossAttention2D(nn.Module):
    def __init__(self, dim_q, dim_kv, heads=8, dropout=0.1):
        super().__init__()
        # 选择能整除的头数
        cand = [8,6,4,3,2,1]
        self.heads = next((h for h in cand if dim_q % h == 0), 1)
        self.scale = (dim_q // self.heads) ** -0.5

        self.to_q = nn.Conv2d(dim_q, dim_q, 1, bias=False)
        # 将 KV 投到与 Q 相同维度，便于分头计算
        self.to_k = nn.Conv2d(dim_kv, dim_q, 1, bias=False)
        self.to_v = nn.Conv2d(dim_kv, dim_q, 1, bias=False)
        self.proj = nn.Conv2d(dim_q, dim_q, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, context):
        """
        x: (B,C,H,W)  -> queries
        context: (B,Ck,H,W) -> keys/values（来自 cond 编码器）
        """
        B, C, H, W = x.shape
        q = self.to_q(x)
        k = self.to_k(context)
        v = self.to_v(context)

        # (B, heads, HW, C_head)
        def reshape_heads(t):
            h = self.heads
            t = t.view(B, h, C // h, H * W).permute(0,1,3,2)  # (B,h,N,d)
            return t

        q = reshape_heads(q)
        k = reshape_heads(k)
        v = reshape_heads(v)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B,h,N,N)
        attn = attn.softmax(dim=-1)
        attn = self.drop(attn)
        out = attn @ v                                    # (B,h,N,d)
        out = out.permute(0,1,3,2).contiguous().view(B, C, H, W)
        return self.proj(out)


# -----------------------------
# 条件 UNet：输入 noisy(3) 与 cond(3) -> 输出 pred_x0(3)
# -----------------------------
class ConditionalUNet(nn.Module):
    def __init__(self, base=128, ch_mult=(1,2,4,8), time_dim=320,
                 in_channels=6, out_channels=3,
                 attn_res={64}, attn_heads=4, attn_dropout=0.0):
        super().__init__()
        self.attn_res = set(attn_res)

        # 时间嵌入
        self.time_mlp = nn.Sequential(SinusoidalPosEmb(time_dim), TimeMLP(time_dim, time_dim))

        # 通道配置
        chans = [base * m for m in ch_mult]
        self.chans = chans

        # 条件编码器：只吃 cond 的 3 通道，输出多尺度特征
        self.cond_enc = CondEncoder(in_ch=3, chans=chans)

        # 输入：仍做 noisy+cond 拼接（稳）
        self.in_conv = conv3x3(in_channels, chans[0])

        # 下采样路径
        self.downs = nn.ModuleList(); self.res_down = nn.ModuleList()
        self.ca_down = nn.ModuleList(); self.proj_down = nn.ModuleList()
        in_c = chans[0]
        for i, c in enumerate(chans):
            self.res_down.append(nn.Sequential(ResBlock(in_c, c, time_dim),
                                               ResBlock(c, c, time_dim)))
            # cross-attn（低分辨率用），否则用 1×1 投影注入
            self.ca_down.append(CrossAttention2D(dim_q=c, dim_kv=c, heads=attn_heads, dropout=attn_dropout))
            self.proj_down.append(nn.Conv2d(c, c, 1))
            in_c = c
            if i < len(chans)-1:
                self.downs.append(Down(in_c))

        # bottleneck
        self.mid = nn.Sequential(ResBlock(in_c, in_c, time_dim),
                                 ResBlock(in_c, in_c, time_dim))
        self.ca_mid   = CrossAttention2D(dim_q=in_c, dim_kv=in_c, heads=attn_heads, dropout=attn_dropout)
        self.proj_mid = nn.Conv2d(in_c, in_c, 1)

        # 上采样路径
        self.ups = nn.ModuleList(); self.res_up = nn.ModuleList()
        self.ca_up = nn.ModuleList(); self.proj_up = nn.ModuleList()
        for i in reversed(range(len(chans))):
            c = chans[i]
            self.res_up.append(nn.Sequential(ResBlock(in_c + c, c, time_dim),
                                             ResBlock(c, c, time_dim)))
            self.ca_up.append(CrossAttention2D(dim_q=c, dim_kv=c, heads=attn_heads, dropout=attn_dropout))
            self.proj_up.append(nn.Conv2d(c, c, 1))
            in_c = c
            if i > 0: self.ups.append(Up(in_c))

        self.out = nn.Sequential(nn.GroupNorm(min(32, in_c), in_c), nn.SiLU(), conv3x3(in_c, out_channels))

    def _inject(self, x, cond_feat, attn_module, proj_module):
        # 仅当当前分辨率在 attn_res 才做 cross-attn；否则做 1×1 投影注入
        if x.shape[-1] in self.attn_res or x.shape[-2] in self.attn_res:
            if cond_feat.shape[2:] != x.shape[2:]:
                cond_feat = F.interpolate(cond_feat, size=x.shape[2:], mode="nearest")
            return x + attn_module(x, cond_feat)
        else:
            if cond_feat.shape[2:] != x.shape[2:]:
                cond_feat = F.interpolate(cond_feat, size=x.shape[2:], mode="nearest")
            return x + proj_module(cond_feat)

    def forward(self, x_noisy, t, cond):
        t_emb = self.time_mlp(t)
        cond_feats = self.cond_enc(cond)

        x = torch.cat([x_noisy, cond], dim=1)
        x = self.in_conv(x)

        feats = []
        # 下采样
        for i, (blk, attn, proj, down) in enumerate(zip(self.res_down, self.ca_down, self.proj_down, list(self.downs)+[None])):
            x = blk[0](x, t_emb); x = blk[1](x, t_emb)
            x = self._inject(x, cond_feats[i], attn, proj)
            feats.append(x)
            if down is not None: x = down(x)

        # bottleneck
        x = self.mid[0](x, t_emb); x = self.mid[1](x, t_emb)
        x = self._inject(x, cond_feats[-1], self.ca_mid, self.proj_mid)

        # 上采样
        for i, (blk, attn, proj, up) in enumerate(zip(self.res_up, self.ca_up, self.proj_up, list(self.ups)+[None])):
            skip = feats.pop()
            x = torch.cat([x, skip], dim=1)
            x = blk[0](x, t_emb); x = blk[1](x, t_emb)
            level_idx = len(self.chans) - 1 - i
            x = self._inject(x, cond_feats[level_idx], attn, proj)
            if up is not None: x = up(x)

        return self.out(x).tanh()


# -----------------------------
# 线性噪声表 & q 采样
# -----------------------------
class DiffusionSchedule:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=2e-2, device="cuda"):
        self.T = T
        betas = torch.linspace(beta_start, beta_end, T, device=device)
        alphas = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)
        self.register(betas, alphas, alpha_bar)

    def register(self, betas, alphas, alpha_bar):
        self.betas = betas
        self.alphas = alphas
        self.alpha_bar = alpha_bar
        self.sqrt_ab = torch.sqrt(alpha_bar)
        self.sqrt_1m_ab = torch.sqrt(1.0 - alpha_bar)
        self.sqrt_a = torch.sqrt(alphas)
        self.inv_sqrt_a = 1.0 / torch.sqrt(alphas)

    def sample_q(self, x0, t, eps=None):
        """
        q(x_t | x0) = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*eps
        t: [B] in [0, T-1]
        """
        B = x0.size(0)
        if eps is None:
            eps = torch.randn_like(x0)
        sqrt_ab_t = self.sqrt_ab[t].view(B, 1, 1, 1)
        sqrt_1m_ab_t = self.sqrt_1m_ab[t].view(B, 1, 1, 1)
        x_t = sqrt_ab_t * x0 + sqrt_1m_ab_t * eps
        return x_t, eps

# -----------------------------
# 采样（DDIM，预测 x0 版本）
# -----------------------------
@torch.no_grad()
def sample_ddim(model, sched, cond, steps=50, eta=0.0):
    device = cond.device
    B, _, H, W = cond.shape
    T = sched.T

    ts = torch.linspace(T-1, 0, steps, device=device).long()
    x = torch.randn(B, 3, H, W, device=device)

    for i, t in enumerate(ts):
        # 当前步索引（长度为 B 的向量）
        tt = torch.full((B,), int(t.item()), device=device, dtype=torch.long)

        pred_x0 = model(x, tt, cond)
        ab_t    = sched.alpha_bar[tt].view(B,1,1,1)
        eps_t   = (x - torch.sqrt(ab_t)*pred_x0) / torch.sqrt(1 - ab_t + 1e-8)

        if i == steps - 1:
            x = pred_x0
            break

        t_next  = ts[min(i+1, steps-1)]
        tt_next = torch.full((B,), int(t_next.item()), device=device, dtype=torch.long)
        ab_next = sched.alpha_bar[tt_next].view(B,1,1,1)

        eta_tensor = torch.tensor(eta, device=device).view(1,1,1,1)
        sigma_t = eta_tensor * torch.sqrt((1 - ab_next)/(1 - ab_t + 1e-8) * (1 - ab_t/ab_next + 1e-8))
        noise = torch.randn_like(x)
        x = torch.sqrt(ab_next)*pred_x0 + torch.sqrt(1 - ab_next - sigma_t**2)*eps_t + sigma_t*noise

    return x.clamp(-1, 1)

# -----------------------------
# 测试集可视化
# -----------------------------
@torch.no_grad()
def save_eval_grid(model, sched, test_loader, save_path, steps=50, eta=0.0, max_vis=8):
    """
    用测试集第一个 batch 生成 [cond | gt | pred] 的可视化网格并保存。
    """
    model.eval()
    try:
        cond, x0 = next(iter(test_loader))
    except StopIteration:
        print("[WARN] 测试集为空，跳过可视化保存")
        return

    device = next(model.parameters()).device
    cond = cond.to(device)
    x0   = x0.to(device)

    vis_n = min(cond.shape[0], max_vis)
    cond_vis = cond[:vis_n]
    x0_vis   = x0[:vis_n]

    pred_vis = sample_ddim(model, sched, cond_vis, steps=steps, eta=eta)

    grid = vutils.make_grid(torch.cat([cond_vis, x0_vis, pred_vis], dim=0), nrow=vis_n)
    Image.fromarray(to_uint8(grid.permute(1,2,0).cpu()).numpy()).save(save_path)
    img = to_uint8(grid.permute(1,2,0).cpu()).numpy()
    plt.figure(figsize=(vis_n*12, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# -----------------------------
# 训练
# -----------------------------
def train(
    data_root="data/regions",
    image_size=512,
    batch_size=2,
    lr=1e-4,
    epochs=20,
    num_workers=4,
    T=1000,
    save_dir="runs/exp1",
    ema_decay=0.999
):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device = {device}")

    train_loader, test_loader = create_data_loaders(
        root=data_root, size=image_size,
        batch_size=batch_size, num_workers=num_workers,
        test_ratio=0.1, seed=42
    )

    model = ConditionalUNet(base=192, ch_mult=(1,2,4,8), time_dim=320, in_channels=6, out_channels=3).to(device)
    model_ema = ConditionalUNet(base=192, ch_mult=(1,2,4,8), time_dim=320, in_channels=6, out_channels=3).to(device)
    model_ema.load_state_dict(model.state_dict())
    for p in model_ema.parameters(): p.requires_grad_(False)

    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = DiffusionSchedule(T=T, device=device)

    for epoch in range(1, epochs+1):
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        model.train()
        for cond, x0 in pbar:
            cond = cond.to(device)  # 漂移图（条件）
            x0   = x0.to(device)    # 原图（监督）
            B = x0.size(0)
            #t = torch.randint(0, T, (B,), device=device).long()
            u = torch.rand(B, device=device)
            t = (u**2 * (T-1)).long()  # 越靠近 0 的概率越大（平方可改成 1.5~3 之间调味）

            # q(x_t | x0)
            x_t, _ = sched.sample_q(x0, t)

            # 预测 x0
            pred_x0 = model(x_t, t, cond)

            # Loss：L1（可自行改为 L2 或混合）
            loss = 0.3 * F.l1_loss(pred_x0, x0) + 0.7 * F.mse_loss(pred_x0, x0)

            optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

            # EMA
            with torch.no_grad():
                for p, q in zip(model.parameters(), model_ema.parameters()):
                    q.data.mul_(ema_decay).add_(p.data, alpha=1-ema_decay)

            pbar.set_postfix(loss=float(loss.item()))

        # 测试集可视化
        if epoch % 5 == 0:
          test_vis_path = os.path.join(save_dir, f"test_epoch_{epoch:03d}.png")
          save_eval_grid(model_ema, sched, test_loader, test_vis_path, steps=100, eta=0.0, max_vis=8)
          save_eval_grid(model_ema, sched, test_loader, test_vis_path, steps=200, eta=0.0, max_vis=8)

        # 保存权重
        torch.save({
            "model": model.state_dict(),
            "ema": model_ema.state_dict(),
            "opt": optim.state_dict(),
            "epoch": epoch
        }, os.path.join("drive/MyDrive/", f"recover1ckpt.pt"))




In [ ]:
# -----------------------------
# 入口
# -----------------------------
if __name__ == "__main__":
    train(
        data_root="drive/MyDrive/resize_vae1",  # 这里换成你的图片目录
        image_size=512,
        batch_size=1,              # 显存不够改成 1
        lr=5e-5,
        epochs=150,
        num_workers=4,
        T=1000,
        save_dir="runs/exp1",
        ema_decay=0.999
    )



In [4]:
import os
import torch
from PIL import Image
from torchvision import transforms

def infer_single(
    model_ckpt_path="drive/MyDrive/recover1ckpt.pt",
    cond_folder="drive/MyDrive/vae",
    save_dir="drive/MyDrive/vae",
    steps=200,
    eta=0.0,
    device="cuda"
):
    os.makedirs(save_dir, exist_ok=True)

    # 加载模型和权重
    model = ConditionalUNet(base=192, ch_mult=(1,2,4,8), time_dim=320, in_channels=6, out_channels=3).to(device)
    sched = DiffusionSchedule(T=1000, device=device)

    ckpt = torch.load(model_ckpt_path, map_location=device)
    model.load_state_dict(ckpt["ema"])
    model.eval()

    # 图像预处理 transform
    tf = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])

    # 遍历文件夹中所有 cond 图
    all_images = sorted([
        f for f in os.listdir(cond_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    for i, filename in enumerate(all_images):
        img_path = os.path.join(cond_folder, filename)
        try:
            img = Image.open(img_path).convert("RGB")
        except:
            print(f"[WARN] 读取失败: {img_path}")
            continue

        cond = tf(img).unsqueeze(0).to(device)  # shape [1,3,H,W]

        with torch.no_grad():
            pred_x0 = sample_ddim(model, sched, cond, steps=steps, eta=eta)  # [1,3,H,W]

        out_img = to_uint8(pred_x0[0].permute(1,2,0).cpu()).numpy()
        save_path = os.path.join(save_dir, f"gen_{i}.png")
        Image.fromarray(out_img).save(save_path)
        print(f"[INFO] 生成保存: {save_path}")

if __name__ == "__main__":
    infer_single()


[INFO] 生成保存: drive/MyDrive/vae/gen_0.png
[WARN] 读取失败: drive/MyDrive/vae/gen_1.png
[WARN] 读取失败: drive/MyDrive/vae/gen_2.png
[WARN] 读取失败: drive/MyDrive/vae/gen_3.png
[WARN] 读取失败: drive/MyDrive/vae/gen_4.png
[WARN] 读取失败: drive/MyDrive/vae/gen_5.png
[WARN] 读取失败: drive/MyDrive/vae/gen_6.png
[WARN] 读取失败: drive/MyDrive/vae/gen_7.png
[INFO] 生成保存: drive/MyDrive/vae/gen_8.png
[INFO] 生成保存: drive/MyDrive/vae/gen_9.png
[INFO] 生成保存: drive/MyDrive/vae/gen_10.png
[INFO] 生成保存: drive/MyDrive/vae/gen_11.png
[INFO] 生成保存: drive/MyDrive/vae/gen_12.png
[INFO] 生成保存: drive/MyDrive/vae/gen_13.png
[INFO] 生成保存: drive/MyDrive/vae/gen_14.png
[INFO] 生成保存: drive/MyDrive/vae/gen_15.png
[INFO] 生成保存: drive/MyDrive/vae/gen_16.png
[INFO] 生成保存: drive/MyDrive/vae/gen_17.png
[INFO] 生成保存: drive/MyDrive/vae/gen_18.png
[INFO] 生成保存: drive/MyDrive/vae/gen_19.png
[INFO] 生成保存: drive/MyDrive/vae/gen_20.png
[INFO] 生成保存: drive/MyDrive/vae/gen_21.png
[INFO] 生成保存: drive/MyDrive/vae/gen_22.png
[INFO] 生成保存: drive/MyDrive/vae/gen_23.png
[I